In [3]:
%pip install tensorflow roboflow scikit-learn matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Cek ketersediaan GPU (Opsional, pastikan TensorFlow mendeteksi GPU jika komputer Anda memilikinya)
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


In [5]:
%pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LsuywI3tHluMjbVVnZn9")
project = rf.workspace("roboflow-universe-projects").project("banana-ripeness-classification")
version = project.version(4)
dataset = version.download("folder")

%mv D:/Perkuliahan/Semester 5/CERTAN/Praktikum/CERTAN_2026/Week10/Banana-Ripeness-Classification-4/valid

Note: you may need to restart the kernel to use updated packages.
loading Roboflow workspace...
loading Roboflow project...


UsageError: Line magic function `%mv` not found.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Direktori dataset
train_dir = dataset.location + "/train"
valid_dir = dataset.location + "/valid"

# Augmentasi data dan preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Normalisasi gambar
    rotation_range=20,      # Rotasi acak
    width_shift_range=0.2,  # Pergeseran horizontal
    height_shift_range=0.2, # Pergeseran vertikal
    shear_range=0.2,        # Perubahan sudut
    zoom_range=0.2,         # Zoom acak
    horizontal_flip=True    # Membalik gambar secara horizontal
)

# Untuk set validasi, kita hanya melakukan rescale
valid_datagen = ImageDataGenerator(rescale=1./255)

# Membuat data generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),   # Ubah ukuran gambar
    batch_size=32,
    class_mode='categorical'  # Klasifikasi multi-kelas
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # Penting saat tahap evaluasi/prediksi!
)


In [6]:
#%%
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Membangun model CNN
model = Sequential()

# Lapisan Convolutional pertama
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lapisan Convolutional kedua
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lapisan Convolutional ketiga
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lapisan Flatten
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))  # Dropout untuk mencegah overfitting

# Lapisan Output (Disesuaikan dengan jumlah kelas pada generator)
num_classes = len(train_generator.class_indices)
model.add(Dense(num_classes, activation='softmax'))

# Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Menampilkan ringkasan model
model.summary()

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.


NameError: name 'train_generator' is not defined

In [ ]:
#%%
# Melatih model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // valid_generator.batch_size,
    epochs=9  # Jumlah epoch
)

In [ ]:
# Model evaluation using precision, recall, confution matrix and f1 score, display confusion matrix in plt

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Evaluasi model pada keseluraun class data validasi
val_loss, val_accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

# Evaluasi model pada setiap class data validasi
y_pred = model.predict(valid_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = valid_generator.classes

# Menghitung metrik evaluasi
report = classification_report(y_true, y_pred_classes)
print(report)

# Membuat confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Menampilkan confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(valid_generator.class_indices))
plt.xticks(tick_marks, valid_generator.class_indices, rotation=45)
plt.yticks(tick_marks, valid_generator.class_indices)

# Menambahkan label pada setiap sel
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Visualisasi akurasi
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.show()

# Visualisasi loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()


In [ ]:
# inference using model, and add text to displayed predict image

from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np

def predict_image(image_path):
  img = image.load_img(image_path, target_size=(224, 224))
  img_array = image.img_to_array(img)
  img_array = np.expand_dims(img_array, axis=0)
  img_array /= 255.

  prediction = model.predict(img_array)
  predicted_class = np.argmax(prediction)

  # Get class labels from the generator
  class_labels = list(train_generator.class_indices.keys())

  # Display the image with prediction
  plt.imshow(img)
  plt.title(f"Predicted label: {class_labels[predicted_class]}")
  plt.axis('off')
  plt.show()

# Example usage
image_path = "D:/Perkuliahan/Semester 5/CERTAN/Praktikum/CERTAN_2026/Week10/Banana-Ripeness-Classification-4/test/ripe/musa-acuminata-banana-ad772802-394a-11ec-996c-d8c4975e38aa_jpg.rf.ac1e17757ffe79f22868f68252821f9d.jpg"
predict_image(image_path)


# YOLOv8 Transfer Learning

## Install YOLOv8

In [ ]:
# Pip install method (recommended)

%pip install ultralytics

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

## Inference with Pre-trained COCO Model

In [ ]:
!yolo task=classify mode=predict model=yolov8x-cls.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg'

In [ ]:
from IPython.display import Image
Image(filename='runs/classify/predict/dog.jpeg', height=600)


## Custom Training

In [ ]:
!yolo task=classify mode=train model=yolov8x-cls.pt data={dataset.location} epochs=5 imgsz=224 batch=32 freeze=5

In [ ]:
# prompt: show image from runs/classify/train/result.png

from IPython.display import Image
Image('/content/runs/classify/train/results.png', width=750)


## Validate Custom Model

In [ ]:
!yolo task=classify mode=val model=/content/runs/classify/train/weights/best.pt data={dataset.location}

In [ ]:
Image('/content/runs/classify/val/confusion_matrix_normalized.png', width=750)

## Inference menggunakan Custom Model

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/overripe

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/ripe


In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/rotten

In [ ]:
!yolo task=classify mode=predict model=/content/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/unripe

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict2/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))



In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict3/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict4/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/classify/predict5/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))